# MaplePy
### Madelung Part of Lattice Energy (MAPLE) for Crystal Structures

Computes per-site partial MAPLE values from crystal structures, following Hoppe's formalism.

**Key references:**
- Hoppe, R. (1966) *Angew. Chem. Int. Ed.* **5**, 95–106 — Madelung constants & partial Madelung constants
- Hoppe, R. (1970) *Angew. Chem. Int. Ed.* **9**, 25–34 — Coordination number, MAPLE, effective CN
- Hoppe, R. (1995) *Z. Naturforsch. A* **50**, 555–567 — MAPLE as solid-state diagnostic tool

**Three validation targets (all must pass before research use):**
1. **NaCl** — PMC(Na⁺) = PMC(Cl⁻) = 0.8738, MF = 1.7476  *(Hoppe 1966)*
2. **TiO₂ polymorphs** — rutile/anatase MAPLE within ~10 kcal/mol  *(Hoppe 1995, Table 5)*
3. **BaTiO₃ additivity** — MAPLE(BaTiO₃) ≈ MAPLE(BaO) + MAPLE(TiO₂), Δ < 1%  *(Hoppe 1995, p. 560)*

---
**Physical basis:** MAPLE is the electrostatic (Madelung) component of the lattice energy.
Unlike the global Madelung constant, *per-site partial MAPLE values* reveal which crystallographic
sites are electrostatically stable or strained — a long-range complement to BVS/GII diagnostics.

## 0. Setup

In [ ]:
# Run once to install dependencies
# !pip install pymatgen pandas numpy

In [ ]:
from pymatgen.core import __version__
print(__version__)

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

from pymatgen.core import Structure, Lattice
from pymatgen.analysis.ewald import EwaldSummation

# ── Physical constants ─────────────────────────────────────────────────────────
EV_TO_KCAL_MOL = 23.0609   # 1 eV (per particle) = 23.0609 kcal/mol (per mole)
KCAL_TO_KJ     = 4.184     # 1 kcal = 4.184 kJ
# MAPLE conversion factor: e²Nₐ / 4πε₀ in units of Å·kcal/mol
# Hoppe uses 331.81 (older constants); modern value is 332.06
MAPLE_FACTOR   = 332.06    # Å·kcal/mol

print("MaplePy loaded.")
print(f"  EV_TO_KCAL_MOL = {EV_TO_KCAL_MOL}")
print(f"  MAPLE_FACTOR   = {MAPLE_FACTOR} Å·kcal/mol  (Hoppe 1995 uses 331.81)")

---
## 1. Core Functions

In [ ]:
# 1. Core Functions (v3)
# ─────────────────────────────────────────────────────────────────────────────

import re as _re
import json
import warnings
warnings.filterwarnings('ignore', message='dict interface is deprecated',
                        category=DeprecationWarning)
warnings.filterwarnings('ignore', message='Issues encountered while parsing CIF',
                        category=UserWarning)
warnings.filterwarnings('ignore', category=DeprecationWarning,
                        module='pymatgen')

# ── Helpers: safe extraction of element symbol and oxi_state ─────────────────
# pymatgen sites can hold either a Species object (ordered) or a Composition
# (disordered). These helpers handle both transparently.

def _site_element(site):
    """
    Return the majority element symbol for a site.
    For mixed-occupancy sites, returns the species with highest occupancy.
    """
    sp = site.species
    if len(sp) > 1:
        majority = max(sp.items(), key=lambda x: x[1])
        el = majority[0]
        return el.element.symbol if hasattr(el, 'element') else el.symbol
    if hasattr(sp, 'elements') and len(sp.elements) > 0:
        el = sp.elements[0]
        return el.symbol if hasattr(el, 'symbol') else str(el)
    return site.specie.element.symbol


def _site_oxi_state(site):
    """
    Return the oxidation state (float) for a site.
    For mixed-occupancy sites, returns the occupancy-weighted mean charge.
    """
    sp = site.species
    if len(sp) > 1:
        total_oxi = 0.0
        total_occ = 0.0
        for species, occ in sp.items():
            if hasattr(species, 'oxi_state'):
                total_oxi += species.oxi_state * occ
                total_occ += occ
        if total_occ > 0:
            return total_oxi / total_occ
        return 0.0
    if hasattr(sp, 'elements') and len(sp.elements) > 0:
        el = sp.elements[0]
        if hasattr(el, 'oxi_state'):
            return float(el.oxi_state)
    return float(site.specie.oxi_state)


def _is_disordered(structure):
    """Check whether any site in the structure has mixed occupancy."""
    for site in structure:
        if len(site.species) > 1:
            return True
    return False


# ── Oxidation state helpers (unchanged) ──────────────────────────────────────

def _load_oxi_json(path):
    """Load the common_oxidation_states.json, returning {} on failure."""
    try:
        with open(path, 'r') as f:
            return json.load(f)
    except Exception:
        return {}


def _load_maple_library(data_dir='.'):
    """Load maple_library.json, returning {} on failure."""
    try:
        with open(Path(data_dir) / 'maple_library.json', 'r') as f:
            return json.load(f)
    except Exception:
        return {}


def _extract_cif_oxidation_states(cif_path):
    """
    Parse _atom_type_oxidation_number loop from a CIF file.

    Mirrors Eir's extract_cif_oxidation_states() — the most reliable source
    for ICSD CIFs, which always embed oxidation states in this loop.

    Returns
    -------
    dict  {element: charge}  e.g. {'Na': 1, 'Cl': -1}, or {} if not found.
    """
    oxidation_states = {}
    try:
        with open(cif_path, 'r', encoding='utf-8', errors='ignore') as f:
            lines = f.readlines()
    except Exception:
        return {}

    in_loop      = False
    columns      = []
    symbol_col   = None
    oxi_col      = None

    for line in lines:
        ls = line.strip()

        if ls == 'loop_':
            in_loop    = False
            columns    = []
            symbol_col = None
            oxi_col    = None
            continue

        if ls.startswith('_atom_type_'):
            idx = len(columns)
            columns.append(ls)
            if ls == '_atom_type_symbol':
                symbol_col = idx
            elif ls == '_atom_type_oxidation_number':
                oxi_col  = idx
                in_loop  = True
            continue

        if in_loop and columns and not ls.startswith('_'):
            if not ls or ls.startswith('#') or ls.startswith('loop_'):
                in_loop = False
                continue
            parts = ls.split()
            if symbol_col is not None and symbol_col < len(parts):
                raw     = parts[symbol_col]
                element = _re.sub(r'[0-9+\-]+', '', raw)
                element = (element.capitalize() if len(element) > 1
                           else element.upper())
                if oxi_col is not None and oxi_col < len(parts):
                    ox_str = parts[oxi_col]
                    if ox_str not in ['?', '.']:
                        try:
                            oxidation_states[element] = round(float(ox_str))
                        except ValueError:
                            pass

    return oxidation_states


def _oxi_from_library(elements, reduced_formula, library):
    """
    Look up oxidation states by reduced formula in maple_library.json.
    Returns dict or None if formula not found or elements not fully covered.
    """
    overrides = library.get('oxidation_state_overrides', {})
    entry = overrides.get(reduced_formula)
    if entry and isinstance(entry, dict):
        if all(el in entry for el in elements):
            return {el: entry[el] for el in elements}
    return None


def _oxi_from_json_fallback(elements, oxi_json):
    """
    Attempt oxidation state assignment using common_oxidation_states.json.
    Returns dict or None if any element is ambiguous (multiple common states).
    """
    assignment = {}
    for el in elements:
        states = oxi_json.get(el, [])
        if len(states) == 1:
            assignment[el] = states[0]
        else:
            return None
    return assignment

# ── load_structure (v4) ──────────────────────────────────────────────────────
#
# KEY CHANGE from v3: pymatgen's native per-site oxidation state assignment
# is now the FIRST choice after user override. ICSD CIFs embed oxidation
# states in the _atom_site_type_symbol column (e.g., Mn3+, Mn4+), and
# pymatgen correctly assigns these per-site, preserving:
#   - Non-integer states (Pr3.8+ for mixed-valence)
#   - Different states on different sites (Mn3+ on site 1, Mn4+ on site 2)
#   - Charge neutrality
#
# The previous approach read element-level values from _atom_type_oxidation_number,
# which collapsed mixed-valence to a single state per element and broke charge
# balance for ~35% of structures. That is now a fallback, not the default.

def _has_native_oxi_states(structure):
    """
    Check whether pymatgen assigned oxidation states when parsing the CIF.
    Returns True if all sites contain Species (not bare Element) objects.
    """
    from pymatgen.core.periodic_table import Species
    for site in structure:
        for sp in site.species:
            if not isinstance(sp, Species):
                return False
    return True


def _native_charge_balance(structure):
    """
    Compute the total charge using pymatgen's native per-site assignments.
    Handles mixed-occupancy sites correctly via occupancy weighting.
    """
    total = 0.0
    for site in structure:
        for sp, occ in site.species.items():
            if hasattr(sp, 'oxi_state'):
                total += sp.oxi_state * occ
    return total


def load_structure(cif_path, oxidation_states=None, data_dir='.'):
    """
    Load a CIF and decorate with oxidation states.

    Mixed-occupancy sites are handled via mean-field charge averaging
    in the downstream compute_maple() function.

    Priority chain for oxidation state assignment:
    1. User-supplied oxidation_states dict          (highest priority)
    2. pymatgen native per-site assignment from CIF (preserves mixed valence,
       non-integer states, and per-site distinctions)
    3. _atom_type_oxidation_number loop in the CIF  (element-level fallback)
    4. maple_library.json formula override           (unambiguous binaries)
    5. common_oxidation_states.json fallback         (single-state elements)
    6. Raises ValueError with a helpful message      (never silently zeros)

    Parameters
    ----------
    cif_path        : str or Path
    oxidation_states: dict, optional  e.g. {'Na': 1, 'Cl': -1}
    data_dir        : str  path to folder containing maple_library.json
                      and common_oxidation_states.json

    Returns
    -------
    pymatgen Structure with oxidation states applied.
    """
    cif_path = str(cif_path)

    # ── Load CIF, catching unrecognised element labels ────────────────────
    try:
        from pymatgen.io.cif import CifParser
        parser = CifParser(cif_path, occupancy_tolerance=1.05)
        structures = parser.get_structures()
        if not structures:
            raise ValueError("No valid structures parsed from CIF.")
        structure = structures[0]
    except Exception as e:
        if "not a valid Element" in str(e):
            raise ValueError(
                f"Unrecognised element label in CIF: {e}. "
                f"Common causes: D (deuterium) in neutron data, "
                f"or generic labels (M, X) for disordered sites."
            ) from e
        raise

    source = None
    disordered = _is_disordered(structure)

    if disordered:
        print(f"  Note: mixed-occupancy sites detected. "
              f"Using mean-field charge approximation.")

    # ── Check for unphysical occupancies ──────────────────────────────────
    for site in structure:
        occ_sum = sum(site.species.values())
        if occ_sum > 1.01:
            print(f"  WARNING: site occupancy sum = {occ_sum:.3f} "
                  f"(>1.0) at {site.species}. "
                  f"Refinement quality issue.")

    # ── Priority 1: user override ─────────────────────────────────────────
    if oxidation_states is not None:
        structure.add_oxidation_state_by_element(oxidation_states)
        source = 'user'

    # ── Priority 2: pymatgen native per-site assignment ───────────────────
    elif _has_native_oxi_states(structure):
        # pymatgen already assigned Species with oxi_states from the CIF.
        # Check charge balance to confirm they are sensible.
        native_q = _native_charge_balance(structure)
        if abs(native_q) < 0.5:
            source = 'CIF (per-site, charge-neutral)'
        else:
            # Native states exist but are not charge-neutral.
            # This happens for genuinely non-stoichiometric compounds.
            # Still use them — they are the best available assignment.
            source = (f'CIF (per-site, Σq={native_q:+.2f})')
        # No action needed: structure already has the correct states.

    else:
        # ── pymatgen did not assign oxi states — fall back ────────────────
        # Extract element set
        elements = set()
        for site in structure:
            for sp in site.species:
                el = (sp.element.symbol if hasattr(sp, 'element')
                      else sp.symbol)
                elements.add(el)

        # ── Priority 3: CIF _atom_type loop (element-level) ──────────────
        cif_oxi = _extract_cif_oxidation_states(cif_path)

        if (cif_oxi
                and all(el in cif_oxi for el in elements)
                and any(v != 0 for v in cif_oxi.values())):
            structure.add_oxidation_state_by_element(cif_oxi)
            source = 'CIF (_atom_type, element-level fallback)'

        else:
            # ── Priority 4: maple_library.json ────────────────────────────
            library = _load_maple_library(data_dir)
            reduced = structure.composition.reduced_formula
            lib_oxi = _oxi_from_library(elements, reduced, library)

            if lib_oxi is not None:
                structure.add_oxidation_state_by_element(lib_oxi)
                source = f'maple_library.json ({reduced})'

            else:
                # ── Priority 5: common_oxidation_states.json ──────────────
                oxi_json = _load_oxi_json(
                    Path(data_dir) / 'common_oxidation_states.json'
                )
                json_oxi = _oxi_from_json_fallback(elements, oxi_json)

                if json_oxi is not None:
                    structure.add_oxidation_state_by_element(json_oxi)
                    source = 'common_oxidation_states.json'

                # ── Priority 6: informative failure ───────────────────────
                else:
                    missing = [el for el in elements
                               if el not in (cif_oxi or {})]
                    multi = [el for el in elements
                             if len(oxi_json.get(el, [])) > 1]
                    raise ValueError(
                        f"Could not determine oxidation states for: "
                        f"{missing + multi}.\n"
                        f"  CIF embedded states found for: "
                        f"{sorted(cif_oxi.keys()) if cif_oxi else 'none'}.\n"
                        f"  Elements with multiple common states: "
                        f"{multi}.\n"
                        f"  Reduced formula '{reduced}' not found in "
                        f"maple_library.json.\n"
                        f"  Solution: pass oxidation_states={{...}} "
                        f"explicitly, e.g.:\n"
                        f"    load_structure(path, oxidation_states="
                        f"{{'Mo': 6, 'O': -2}})"
                    )

    # ── Charge-balance check ──────────────────────────────────────────────
    total_charge = _native_charge_balance(structure)
    if abs(total_charge) > 0.05:
        print(f"  WARNING: not charge-neutral "
              f"(Σq = {total_charge:+.3f} e).\n"
              f"  Source: {source}. Check oxidation states before "
              f"using MAPLE results.")
    else:
        # Build display of oxi states per element
        oxi_display = {}
        for s in structure:
            el = _site_element(s)
            oxi = _site_oxi_state(s)
            if el in oxi_display:
                # Multiple sites with different oxi states — show range
                if isinstance(oxi_display[el], list):
                    if oxi not in oxi_display[el]:
                        oxi_display[el].append(oxi)
                else:
                    if oxi != oxi_display[el]:
                        oxi_display[el] = [oxi_display[el], oxi]
            else:
                oxi_display[el] = oxi

        parts = []
        for el in sorted(oxi_display.keys()):
            v = oxi_display[el]
            if isinstance(v, list):
                vals = '/'.join(f"{x:+.1f}" if x != int(x)
                               else f"{int(x):+d}" for x in sorted(v))
                parts.append(f"{el}={vals}")
            else:
                if v != int(v):
                    parts.append(f"{el}={v:+.2f}")
                else:
                    parts.append(f"{el}={int(v):+d}")

        print(f"  Oxidation states ({source}): " + ", ".join(parts))

    return structure


# ── _find_d_ref ──────────────────────────────────────────────────────────────

def _find_d_ref(structure):
    """
    Shortest cation-anion distance in the structure.

    This is Hoppe's reference distance d₁ (or d_KA) — the normalisation
    distance on which the Madelung constant and MAPLE are based.
    """
    cat_idx = [i for i, s in enumerate(structure)
               if _site_oxi_state(s) > 0]
    ani_idx = [i for i, s in enumerate(structure)
               if _site_oxi_state(s) < 0]

    if not cat_idx or not ani_idx:
        return min(structure.get_distance(0, j)
                   for j in range(1, len(structure)))

    d_min = np.inf
    for ci in cat_idx:
        for ai in ani_idx:
            d = structure.get_distance(ci, ai)
            if d < d_min:
                d_min = d
    return float(d_min)


# ── SG 62 origin-choice correction ───────────────────────────────────────

def _fix_sg62_origin(structure, verbose=True):
    """
    Fix the Pnma/Pbnm (SG 62) origin-choice artefact for GdFeO3-type
    perovskites.

    Some ICSD CIF entries for SG 62 perovskites use an alternative
    origin choice that places the 8d oxygen (O2) at a symmetry-equivalent
    but non-standard representative position, shifted by (½, ½, 0) from
    the conventional choice.  pymatgen's CifParser applies the listed
    symmetry operations faithfully, producing a physically valid but
    *inequivalent* expanded structure whose Ewald energy differs from
    that of the standard origin.

    Applicable only to SG 62 structures with exactly 20 sites (Z = 4,
    ABO₃ perovskite).  The method identifies O2 sites (Wyckoff 8d,
    z ∉ {¼, ¾}) and constructs an alternative structure with these sites
    shifted by (½, ½, 0).  The Ewald energy of both versions is compared;
    the structure with the more negative energy (higher |MAPLE|) is
    returned, guaranteeing the correct origin is always selected.

    Typical correction magnitude: 6–8 % of MAPLE.

    Parameters
    ----------
    structure : pymatgen Structure with oxidation states assigned
    verbose   : bool

    Returns
    -------
    structure : corrected Structure (or original if no fix needed)
    """
    try:
        _, sg_number = structure.get_space_group_info(symprec=0.1)
    except Exception:
        return structure

    if sg_number != 62 or len(structure) != 20:
        return structure

    # ── Identify O2 (Wyckoff 8d) sites ────────────────────────────────
    # In GdFeO3-type perovskites, O1 occupies 4c (z = ¼ or ¾) and
    # O2 occupies 8d (general z).
    o2_mask = []
    for site in structure:
        is_oxygen = _site_element(site) == 'O'
        z = site.frac_coords[2]
        is_8d = abs(z - 0.25) > 0.1 and abs(z - 0.75) > 0.1
        o2_mask.append(is_oxygen and is_8d)

    if sum(o2_mask) != 8:
        return structure  # Not a standard ABO₃ perovskite layout

    # ── Build alternative with O2 shifted by (½, ½, 0) ────────────────
    new_coords = []
    for i, site in enumerate(structure):
        fc = site.frac_coords.copy()
        if o2_mask[i]:
            fc[0] = (fc[0] + 0.5) % 1.0
            fc[1] = (fc[1] + 0.5) % 1.0
        new_coords.append(fc)

    alt_structure = Structure(
        structure.lattice,
        [site.species for site in structure],
        new_coords,
    )

    # ── Compare Ewald energies ────────────────────────────────────────
    try:
        ew_orig = EwaldSummation(structure).total_energy
        ew_alt  = EwaldSummation(alt_structure).total_energy
    except Exception:
        return structure

    if ew_alt < ew_orig:          # alt is more stable → correct origin
        if verbose:
            delta_pct = abs(ew_alt - ew_orig) / abs(ew_orig) * 100
            print(f"  Applied SG 62 origin-choice correction "
                  f"(O₂ shifted by (½,½,0), ΔE = {delta_pct:.1f}%).")
        return alt_structure

    return structure


# ── compute_maple ────────────────────────────────────────────────────────────

def compute_maple(structure, verbose=True):
    """
    Compute partial MAPLE values for all sites in the structure.

    Uses pymatgen's EwaldSummation for the electrostatic energy.
    Per-site contributions are the row sums of the energy matrix,
    which sum to the total Ewald energy without double-counting.

    For mixed-occupancy sites, the occupancy-weighted mean charge is used.

    Sign convention
    ---------------
    pymatgen energies are negative (bound state).
    MAPLE is reported positive, following Hoppe's convention.

    Parameters
    ----------
    structure : pymatgen Structure with oxidation states
    verbose   : bool

    Returns
    -------
    dict with keys:
        total_maple_kcal, per_site, Z, d_ref, madelung_factor,
        disordered, structure
    """
    disordered = _is_disordered(structure)

    # ── SG 62 origin-choice correction ────────────────────────────────────
    structure = _fix_sg62_origin(structure, verbose=verbose)

    # ── Ewald summation ───────────────────────────────────────────────────
    ewald = EwaldSummation(structure)
    energy_matrix = ewald.total_energy_matrix
    per_site_eV   = energy_matrix.sum(axis=1)

    # ── Formula units per cell (Z) ────────────────────────────────────────
    _, Z_float = structure.composition.get_reduced_composition_and_factor()
    Z = int(round(Z_float))

    # ── Conversion: eV/site → kcal/mol/site ──────────────────────────────
    per_site_maple = np.abs(per_site_eV) * EV_TO_KCAL_MOL

    # ── Total MAPLE per formula unit ──────────────────────────────────────
    total_maple_kcal = np.abs(ewald.total_energy) * EV_TO_KCAL_MOL / Z

    # ── Reference distance ────────────────────────────────────────────────
    d_ref = _find_d_ref(structure)

    # ── Per-site data ─────────────────────────────────────────────────────
    per_site_data = []
    for i, site in enumerate(structure):
        oxi     = _site_oxi_state(site)
        maple_i = float(per_site_maple[i])

        pmc = (maple_i * d_ref / (MAPLE_FACTOR * abs(oxi))
               ) if oxi != 0 else None

        reduced = (maple_i / (oxi ** 2) / d_ref
                   ) if oxi != 0 else None

        per_site_data.append({
            "site_index"      : i,
            "element"         : _site_element(site),
            "oxidation_state" : round(oxi, 4),
            "mixed_site"      : len(site.species) > 1,
            "frac_x"          : round(float(site.frac_coords[0]), 6),
            "frac_y"          : round(float(site.frac_coords[1]), 6),
            "frac_z"          : round(float(site.frac_coords[2]), 6),
            "maple_kcal_mol"  : round(maple_i, 3),
            "pmc"             : round(pmc, 5)     if pmc is not None else None,
            "reduced_maple"   : round(reduced, 4) if reduced is not None else None,
        })

    # ── Madelung factor ───────────────────────────────────────────────────
    try:
        oxi_states = [_site_oxi_state(s) for s in structure]
        z_pos = [z for z in oxi_states if z > 0]
        z_neg = [z for z in oxi_states if z < 0]
        if not z_pos or not z_neg:
            madelung_factor = None
        else:
            mean_zp = np.mean(z_pos)
            mean_zn = abs(np.mean(z_neg))
            madelung_factor = (total_maple_kcal * d_ref
                               / (MAPLE_FACTOR * mean_zp * mean_zn))
    except Exception:
        madelung_factor = None

    result = {
        "total_maple_kcal" : round(total_maple_kcal, 2),
        "per_site"         : per_site_data,
        "Z"                : Z,
        "d_ref"            : round(d_ref, 4),
        "madelung_factor"  : (round(madelung_factor, 4)
                              if madelung_factor is not None else None),
        "disordered"       : disordered,
        "structure"        : structure,
    }

    if verbose:
        comp = structure.composition.reduced_formula
        sg   = structure.get_space_group_info()[0]
        tag  = "  [MEAN-FIELD]" if disordered else ""
        print(f"\n{'─'*60}")
        print(f"  Compound  : {comp}  ({sg}){tag}")
        print(f"  Sites     : {len(structure)}  |  Z = {Z}")
        print(f"  d_ref     : {d_ref:.4f} Å")
        mf_str = f"{madelung_factor:.4f}" if madelung_factor is not None else "n/a"
        print(f"  MF        : {mf_str}")
        print(f"  MAPLE     : {total_maple_kcal:.1f} kcal/mol  "
              f"({total_maple_kcal * KCAL_TO_KJ:.1f} kJ/mol)")
        print(f"{'─'*60}")

    return result


# ── results_to_dataframe ─────────────────────────────────────────────────────

def results_to_dataframe(maple_result):
    """Convert compute_maple() output to a tidy pandas DataFrame."""
    df = pd.DataFrame(maple_result["per_site"])

    comp = maple_result["structure"].composition.reduced_formula
    sg   = maple_result["structure"].get_space_group_info()[0]
    df["compound"]        = comp
    df["space_group"]     = sg
    df["Z"]               = maple_result["Z"]
    df["d_ref_ang"]       = maple_result["d_ref"]
    df["madelung_factor"] = maple_result["madelung_factor"]
    df["total_maple"]     = maple_result["total_maple_kcal"]
    df["disordered"]      = maple_result["disordered"]

    cols = ["compound", "space_group", "Z", "d_ref_ang", "madelung_factor",
            "total_maple", "disordered", "site_index", "element",
            "oxidation_state", "mixed_site",
            "frac_x", "frac_y", "frac_z",
            "maple_kcal_mol", "pmc", "reduced_maple"]
    return df[cols]

---
## 2. Validation

**All three checks must pass before using MaplePy for research.**
These are the exact values Hoppe reports; any significant deviation indicates
an implementation error.

### 2.1  NaCl — Partial Madelung constants

**Expected (Hoppe 1966):**
- PMC(Na⁺) = PMC(Cl⁻) = **0.8738** (rock salt has commutative sublattices → equal PMC)
- Total MF = **1.7476**
- The symmetry equality PMC(Na) = PMC(Cl) is an exact result, not a tolerance question.

In [ ]:
# NaCl: Fm-3m, a = 5.6402 Å (experimental ambient, ICSD)
nacl = Structure.from_spacegroup(
    "Fm-3m",
    Lattice.cubic(5.6402),
    ["Na", "Cl"],
    [[0.0, 0.0, 0.0], [0.5, 0.5, 0.5]]
)
nacl.add_oxidation_state_by_element({"Na": 1, "Cl": -1})

result_nacl = compute_maple(nacl)
df_nacl = results_to_dataframe(result_nacl)
print(df_nacl[["element","oxidation_state","maple_kcal_mol","pmc","reduced_maple"]].to_string(index=False))

In [ ]:
# ── Validation checks ──────────────────────────────────────────────────────────
pmc_na = df_nacl[df_nacl["element"] == "Na"]["pmc"].mean()
pmc_cl = df_nacl[df_nacl["element"] == "Cl"]["pmc"].mean()
mf     = result_nacl["madelung_factor"]

TARGET_PMC = 0.8738
TARGET_MF  = 1.7476
TOL        = 0.005  # 0.5%

print("── NaCl Validation ───────────────────────────────────────────────────────")
print(f"  PMC(Na⁺)   = {pmc_na:.4f}  target {TARGET_PMC}  Δ = {abs(pmc_na-TARGET_PMC):.4f}")
print(f"  PMC(Cl⁻)   = {pmc_cl:.4f}  target {TARGET_PMC}  Δ = {abs(pmc_cl-TARGET_PMC):.4f}")
print(f"  MF (total) = {mf:.4f}  target {TARGET_MF}  Δ = {abs(mf-TARGET_MF):.4f}")
print(f"  Symmetry   |PMC_Na - PMC_Cl| = {abs(pmc_na-pmc_cl):.2e}  (must be ~0)")

pass_na  = abs(pmc_na - TARGET_PMC) < TOL
pass_cl  = abs(pmc_cl - TARGET_PMC) < TOL
pass_mf  = abs(mf     - TARGET_MF)  < TOL
pass_sym = abs(pmc_na - pmc_cl)     < 1e-4

overall = all([pass_na, pass_cl, pass_mf, pass_sym])
print(f"\n  PMC(Na) : {'✓ PASS' if pass_na  else '✗ FAIL'}")
print(f"  PMC(Cl) : {'✓ PASS' if pass_cl  else '✗ FAIL'}")
print(f"  MF      : {'✓ PASS' if pass_mf  else '✗ FAIL'}")
print(f"  Symmetry: {'✓ PASS' if pass_sym else '✗ FAIL'}")
print(f"\n  {'✅  NaCl VALIDATION PASSED' if overall else '❌  NaCl VALIDATION FAILED — check implementation'}")

### 2.2  TiO₂ polymorphs — Near-degenerate modifications

**Expected (Hoppe 1995, Table 5):**
| Polymorph | MAPLE (kcal/mol) |
|-----------|:---------------:|
| Rutile    | 3254.6          |
| Anatase   | 3260.7          |
| Brookite  | 3253.7          |

The key result: despite radically different geometries, MAPLE values differ by only
~7 kcal/mol. This is a non-trivial consequence of the additivity theorem and is what
MAPLE is designed to reveal.

In [ ]:
# Rutile TiO₂: P4₂/mnm (No. 136), a = 4.5933 Å, c = 2.9592 Å
# Ti: 2a (0,0,0); O: 4f (u,u,0) with u = 0.3048
rutile = Structure.from_spacegroup(
    136,
    Lattice.tetragonal(4.5933, 2.9592),
    ["Ti", "O"],
    [[0.0, 0.0, 0.0], [0.3048, 0.3048, 0.0]]
)
rutile.add_oxidation_state_by_element({"Ti": 4, "O": -2})

# Anatase TiO₂: I4₁/amd (No. 141), a = 3.7845 Å, c = 9.5143 Å
# Ti: 4a; O: 8e (0,0,z) with z = 0.2082
anatase = Structure.from_spacegroup(
    141,
    Lattice.tetragonal(3.7845, 9.5143),
    ["Ti", "O"],
    [[0.0, 0.0, 0.0], [0.0, 0.0, 0.2082]]
)
anatase.add_oxidation_state_by_element({"Ti": 4, "O": -2})

print("Rutile  :", rutile.composition,  " | sites:", len(rutile))
print("Anatase :", anatase.composition, " | sites:", len(anatase))

In [ ]:
result_rutile  = compute_maple(rutile,  verbose=True)
result_anatase = compute_maple(anatase, verbose=True)

In [ ]:
maple_r = result_rutile["total_maple_kcal"]
maple_a = result_anatase["total_maple_kcal"]
delta   = abs(maple_r - maple_a)

print("── TiO₂ Validation ───────────────────────────────────────────────────────")
print(f"  MAPLE(rutile)  = {maple_r:.1f}  kcal/mol  (Hoppe: 3254.6)")
print(f"  MAPLE(anatase) = {maple_a:.1f}  kcal/mol  (Hoppe: 3260.7)")
print(f"  |Δ| rutile–anatase = {delta:.1f} kcal/mol")

# Tolerances: 50 kcal/mol absolute (allows for slightly different CIF sources),
# and < 15 kcal/mol inter-polymorph (the near-degeneracy is the key result)
pass_r    = abs(maple_r - 3254.6) < 50
pass_a    = abs(maple_a - 3260.7) < 50
pass_diff = delta < 15

overall = all([pass_r, pass_a, pass_diff])
print(f"\n  Rutile  vs Hoppe 3254.6  : {'✓ PASS' if pass_r    else '✗ FAIL'}  (Δ={abs(maple_r-3254.6):.1f})")
print(f"  Anatase vs Hoppe 3260.7  : {'✓ PASS' if pass_a    else '✗ FAIL'}  (Δ={abs(maple_a-3260.7):.1f})")
print(f"  Polymorphs within 15 kcal: {'✓ PASS' if pass_diff else '✗ FAIL'}  (Δ={delta:.1f})")
print(f"\n  {'✅  TiO₂ VALIDATION PASSED' if overall else '❌  TiO₂ VALIDATION FAILED'}")

### 2.3  BaTiO₃ — Additivity theorem

**Hoppe's additivity theorem:** MAPLE(polynary) = Σ MAPLE(binary constituents)

**Expected (Hoppe 1995, p. 560):**
- MAPLE(BaO) + MAPLE(rutile-TiO₂) = 4092 kcal/mol
- MAPLE(BaTiO₃, cubic) = 4095 kcal/mol
- Δ = 3 kcal/mol = **0.07%**

This near-exact agreement across the binary → ternary transition is the fundamental
validation of MAPLE as a comparative structural tool.

In [ ]:
# BaO: Fm-3m (rock salt), a = 5.5391 Å
bao = Structure.from_spacegroup(
    "Fm-3m",
    Lattice.cubic(5.5391),
    ["Ba", "O"],
    [[0.0, 0.0, 0.0], [0.5, 0.5, 0.5]]
)
bao.add_oxidation_state_by_element({"Ba": 2, "O": -2})

# Cubic BaTiO₃: Pm-3m (No. 221), a = 4.0019 Å
# Ba: 1a (0,0,0)  Ti: 1b (½,½,½)  O: 3c (½,½,0)
batibo3 = Structure.from_spacegroup(
    "Pm-3m",
    Lattice.cubic(4.0019),
    ["Ba", "Ti", "O"],
    [[0.0, 0.0, 0.0], [0.5, 0.5, 0.5], [0.5, 0.5, 0.0]]
)
batibo3.add_oxidation_state_by_element({"Ba": 2, "Ti": 4, "O": -2})

print("BaO:    ", bao.composition,    "| sites:", len(bao))
print("BaTiO₃: ", batibo3.composition, "| sites:", len(batibo3))

In [ ]:
result_bao    = compute_maple(bao,    verbose=True)
result_batio3 = compute_maple(batibo3, verbose=True)

In [ ]:
maple_bao    = result_bao["total_maple_kcal"]
maple_tio2   = result_rutile["total_maple_kcal"]   # from validation 2.2
maple_batio3 = result_batio3["total_maple_kcal"]

maple_sum  = maple_bao + maple_tio2
delta_abs  = abs(maple_batio3 - maple_sum)
delta_pct  = 100 * delta_abs / maple_batio3

print("── BaTiO₃ Additivity Validation ──────────────────────────────────────────")
print(f"  MAPLE(BaO)             = {maple_bao:.1f}  kcal/mol")
print(f"  MAPLE(TiO₂, rutile)    = {maple_tio2:.1f}  kcal/mol")
print(f"  Binary sum             = {maple_sum:.1f}  kcal/mol  (Hoppe: 4092)")
print(f"  MAPLE(BaTiO₃, cubic)   = {maple_batio3:.1f}  kcal/mol  (Hoppe: 4095)")
print(f"  Residual |Δ|           = {delta_abs:.1f} kcal/mol = {delta_pct:.3f}%")

pass_add = delta_pct < 1.0
print(f"\n  Additivity < 1%  : {'✓ PASS' if pass_add else '✗ FAIL'}  ({delta_pct:.3f}%)")
print(f"  {'✅  ADDITIVITY VALIDATION PASSED' if pass_add else '❌  ADDITIVITY VALIDATION FAILED'}")

---
## 3. Single CIF Usage

In [ ]:
# Replace with your CIF path and oxidation states if not embedded in the CIF.
#
# structure = load_structure(
#     "path/to/structure.cif",
#     oxidation_states={"Ca": 2, "O": -2}   # omit if CIF has them
# )
# result = compute_maple(structure)
# df     = results_to_dataframe(result)
# display(df)
# df.to_csv("maple_output.csv", index=False)
#
# ── Demo on already-computed NaCl result ───────────────────────────────────────
df_demo = results_to_dataframe(result_nacl)
display(df_demo)

---
## 4. Additivity Checker

General-purpose function for any polynary compound.

In [ ]:
def check_additivity(polynary_result, binary_results_and_coeffs, verbose=True):
    """
    Check Hoppe's additivity theorem: MAPLE(polynary) ≈ Σ coeff_i × MAPLE(binary_i)

    Parameters
    ----------
    polynary_result : dict from compute_maple()
    binary_results_and_coeffs : list of (result_dict, float) tuples
        e.g. [(result_bao, 1), (result_tio2, 1)] for BaTiO₃ = BaO + TiO₂

    Returns
    -------
    dict  with keys: maple_polynary, maple_binary_sum, delta_abs, delta_pct
    """
    maple_poly = polynary_result["total_maple_kcal"]
    maple_sum  = sum(c * r["total_maple_kcal"] for r, c in binary_results_and_coeffs)

    delta_abs = abs(maple_poly - maple_sum)
    delta_pct = 100 * delta_abs / maple_poly if maple_poly else 0

    if verbose:
        comp = polynary_result["structure"].composition.reduced_formula
        print(f"Additivity: {comp}")
        for r, c in binary_results_and_coeffs:
            bc = r["structure"].composition.reduced_formula
            print(f"  {c:+.0f} × MAPLE({bc}) = {c * r['total_maple_kcal']:.1f} kcal/mol")
        print(f"  Binary sum   = {maple_sum:.1f} kcal/mol")
        print(f"  Polynary     = {maple_poly:.1f} kcal/mol")
        print(f"  |Δ|          = {delta_abs:.1f} kcal/mol = {delta_pct:.3f}%")
        status = ("✓ GOOD (<1%)" if delta_pct < 1.0 else
                  "⚠ MARGINAL (1–2%)" if delta_pct < 2.0 else
                  "✗ POOR (>2%)")
        print(f"  Additivity   : {status}")

    return {"maple_polynary": maple_poly, "maple_binary_sum": maple_sum,
            "delta_abs": delta_abs, "delta_pct": delta_pct}

In [ ]:
# Example: BaTiO₃ using results already computed in validation
check_additivity(result_batio3, [(result_bao, 1), (result_rutile, 1)])

---
## 5. Batch Processing

In [ ]:
# 5. Batch Processing (v3)
# ─────────────────────────────────────────────────────────────────────────────

def batch_maple(cif_dir, output_dir=None, oxidation_states_map=None,
                data_dir='.', verbose=False):
    """
    Process all CIFs in a directory.

    Parameters
    ----------
    cif_dir : str or Path
        Directory containing CIF files.
    output_dir : str or Path, optional
        Directory for output CSVs. If None, creates 'outputs' inside cif_dir.
    oxidation_states_map : dict  {filename_stem: {element: oxi_state}}
        For CIFs without embedded oxidation states.
    data_dir : str
        Path to folder containing common_oxidation_states.json.
    verbose : bool
        If True, print full MAPLE output per structure.

    Returns
    -------
    pd.DataFrame, one row per site per structure
    """
    cif_dir   = Path(cif_dir)
    cif_files = sorted(cif_dir.glob("*.cif"))
    if not cif_files:
        print(f"No CIF files found in {cif_dir}")
        return pd.DataFrame()

    # ── Create output directory ───────────────────────────────────────────
    if output_dir is None:
        from datetime import datetime
        timestamp  = datetime.now().strftime('%Y%m%d_%H%M%S')
        output_dir = cif_dir / f'outputs_{timestamp}'
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    print(f"Processing {len(cif_files)} CIFs in {cif_dir}")
    print(f"Output directory: {output_dir}")
    dfs, failed = [], []

    for cif_path in cif_files:
        oxi = (oxidation_states_map or {}).get(cif_path.stem)
        try:
            s      = load_structure(cif_path, oxidation_states=oxi,
                                    data_dir=data_dir)
            result = compute_maple(s, verbose=verbose)
            df_i   = results_to_dataframe(result)
            df_i["source_cif"] = cif_path.name
            dfs.append(df_i)
            tag = " [MF]" if result["disordered"] else ""
            print(f"  ✓ {cif_path.name:45s}  "
                  f"MAPLE = {result['total_maple_kcal']:.1f} kcal/mol{tag}")
        except Exception as e:
            failed.append((cif_path.name, str(e)))
            print(f"  ✗ {cif_path.name:45s}  {e}")

    # ── Save results ──────────────────────────────────────────────────────
    if dfs:
        master = pd.concat(dfs, ignore_index=True)
        results_csv = output_dir / 'maple_results.csv'
        master.to_csv(results_csv, index=False)
        print(f"\nSaved {len(master)} rows → {results_csv}")
    else:
        master = pd.DataFrame()
        print("\nNo structures processed successfully.")

    if failed:
        failed_df = pd.DataFrame(failed, columns=['cif_file', 'error'])
        failed_csv = output_dir / 'maple_failed.csv'
        failed_df.to_csv(failed_csv, index=False)
        print(f"Saved {len(failed)} failures → {failed_csv}")

    # ── Summary ───────────────────────────────────────────────────────────
    n_total = len(cif_files)
    n_ok    = len(dfs)
    n_fail  = len(failed)
    n_ordered    = sum(1 for d in dfs
                       if not d["disordered"].iloc[0]) if dfs else 0
    n_disordered = n_ok - n_ordered

    print(f"\n{'='*60}")
    print(f"  Total CIFs        : {n_total}")
    print(f"  Processed (OK)    : {n_ok}  "
          f"({n_ordered} ordered, {n_disordered} mean-field)")
    print(f"  Failed            : {n_fail}")
    print(f"{'='*60}")

    return master


# Usage:
# df = batch_maple(Path("/path/to/cifs"))
# df = batch_maple(Path("/path/to/cifs"), output_dir=Path("/custom/output"))

---
## 6. Application: MAPLE vs BVS for Alkaline Earth Oxides

The primary research motivation: test whether MAPLE site potentials correlate with
the systematic BVS deviations in the MgO → CaO → SrO → BaO series (Figure 1,
Reeves-McLaren 2026, *Dalton Trans.*).

**Hypothesis:** If MAPLE(M²⁺) scales linearly with cation radius while BVS deviation
also scales linearly (R² = 0.996 established), the BVS functional form is missing
long-range electrostatic contributions rather than being fundamentally wrong in its
short-range parameterisation. This would sharpen the "parameter revision required"
conclusion and point towards a specific correction strategy.

In [ ]:
# Rock-salt MO structures from experimental ambient ICSD parameters
ae_params = {
    "MgO": {"a": 4.2112, "cation": "Mg", "z": 2, "r_shannon": 0.720},
    "CaO": {"a": 4.8105, "cation": "Ca", "z": 2, "r_shannon": 1.000},
    "SrO": {"a": 5.1408, "cation": "Sr", "z": 2, "r_shannon": 1.180},
    "BaO": {"a": 5.5391, "cation": "Ba", "z": 2, "r_shannon": 1.350},
}

ae_results = {}
for name, p in ae_params.items():
    s = Structure.from_spacegroup(
        "Fm-3m", Lattice.cubic(p["a"]),
        [p["cation"], "O"], [[0,0,0],[0.5,0.5,0.5]]
    )
    s.add_oxidation_state_by_element({p["cation"]: p["z"], "O": -2})
    ae_results[name] = compute_maple(s, verbose=False)

    r = p["r_shannon"]
    m = ae_results[name]["total_maple_kcal"]
    d = ae_results[name]["d_ref"]
    print(f"  {name:4s}  r(M²⁺)={r:.3f}Å  d_ref={d:.4f}Å  MAPLE={m:.1f} kcal/mol")

In [ ]:
# Per-site MAPLE for the cation series
rows = []
for name, result in ae_results.items():
    p   = ae_params[name]
    df_i = pd.DataFrame(result["per_site"])
    maple_cat = df_i[df_i["element"] == p["cation"]]["maple_kcal_mol"].mean()
    maple_ani = df_i[df_i["element"] == "O"]["maple_kcal_mol"].mean()
    pmc_cat   = df_i[df_i["element"] == p["cation"]]["pmc"].mean()
    rows.append({
        "oxide"          : name,
        "cation"         : p["cation"],
        "shannon_r_ang"  : p["r_shannon"],
        "d_ref_ang"      : result["d_ref"],
        "madelung_factor": result["madelung_factor"],
        "maple_total"    : result["total_maple_kcal"],
        "maple_cation"   : round(maple_cat, 2),
        "maple_anion"    : round(maple_ani, 2),
        "pmc_cation"     : round(pmc_cat, 4),
    })

df_ae = pd.DataFrame(rows)
print(df_ae.to_string(index=False))

# Save for merge with BVS deviation data from Eir output
# df_ae.to_csv("ae_oxide_maple.csv", index=False)

In [ ]:
# Quick correlation check: does MAPLE(M²⁺) correlate with Shannon radius?
from numpy.polynomial import polynomial as P

r_vals    = df_ae["shannon_r_ang"].values
maple_cat = df_ae["maple_cation"].values
maple_tot = df_ae["maple_total"].values

# Linear fit
coeffs_cat = np.polyfit(r_vals, maple_cat, 1)
coeffs_tot = np.polyfit(r_vals, maple_tot, 1)

# R²
def r_squared(y, y_fit):
    ss_res = np.sum((y - y_fit)**2)
    ss_tot = np.sum((y - y.mean())**2)
    return 1 - ss_res/ss_tot

r2_cat = r_squared(maple_cat, np.polyval(coeffs_cat, r_vals))
r2_tot = r_squared(maple_tot, np.polyval(coeffs_tot, r_vals))

print("── MAPLE vs Shannon radius (alkaline earth oxides) ───────────────────────")
print(f"  MAPLE(cation) ~ radius:  slope={coeffs_cat[0]:.1f} kcal/mol/Å  R²={r2_cat:.4f}")
print(f"  MAPLE(total)  ~ radius:  slope={coeffs_tot[0]:.1f} kcal/mol/Å  R²={r2_tot:.4f}")
print()
print("Compare with BVS result from Reeves-McLaren (2026):")
print("  BVS deviation ~ radius:  slope ≈ -30 %/Å  R² = 0.996")
print()
print("If MAPLE(cation) also scales linearly (R² >> 0), this supports the")
print("hypothesis that BVS is missing long-range electrostatic contributions.")

---
## 7. GUI: Load and Process Real CIFs

Tkinter file picker → compute MAPLE for each selected CIF → display results inline → save CSV.

**Oxidation states:** Edit the dict in the first cell below before launching.
ICSD CIFs for simple binary oxides and halides usually embed oxidation states and
the dict can be left empty (`{}`). For CIFs without embedded states, or where
pymatgen's auto-assignment is unreliable (transition metals, mixed valence),
provide explicit values keyed by element symbol.

The output CSV uses the same column schema as the programmatic results above,
so it can be merged directly with Eir output for BVS/MAPLE comparison.

In [ ]:
# 7. GUI: Load and Process Real CIFs (v3)
# ─────────────────────────────────────────────────────────────────────────────
#
# Tkinter file picker → compute MAPLE for each selected CIF → display
# results inline → save CSV files to 'outputs' subdirectory.
#
# Three output files are created in an 'outputs' folder alongside the CIFs:
#   maple_results.csv  — full site-level data (every site, every structure)
#   maple_summary.csv  — one row per structure (compound, SG, MAPLE, etc.)
#   maple_failed.csv   — every failed CIF with its error message


# ── User-editable oxidation state overrides ───────────────────────────────────
# Leave empty {} to use CIF-embedded states or pymatgen auto-assignment.
# Add entries only for elements where you want to override, e.g.:
#   OXIDATION_STATES = {"Ba": 2, "O": -2}

OXIDATION_STATES = {}

# ── Data directory ────────────────────────────────────────────────────────────
# Folder containing common_oxidation_states.json (and shannon_radii.json).
DATA_DIR = "."

print("Oxidation state overrides:",
      OXIDATION_STATES if OXIDATION_STATES else
      "(none — will use CIF or auto-assign)")


# ── GUI function ──────────────────────────────────────────────────────────────

import tkinter as tk
from tkinter import filedialog
import traceback

def launch_maple_gui():
    """
    Open a file picker, process selected CIFs through compute_maple(),
    display results as a DataFrame, and save outputs to an 'outputs'
    subdirectory alongside the selected CIFs.
    """

    # ── File picker ───────────────────────────────────────────────────────
    root = tk.Tk()
    root.withdraw()
    root.attributes('-topmost', True)

    cif_paths = filedialog.askopenfilenames(
        title="Select CIF files for MAPLE analysis",
        filetypes=[("CIF files", "*.cif"), ("All files", "*.*")],
        parent=root
    )
    root.destroy()

    if not cif_paths:
        print("No files selected.")
        return None

    # ── Determine output directory ────────────────────────────────────────
    from datetime import datetime
    cif_parent = Path(cif_paths[0]).parent
    timestamp  = datetime.now().strftime('%Y%m%d_%H%M%S')
    output_dir = cif_parent / f'outputs_{timestamp}'
    output_dir.mkdir(parents=True, exist_ok=True)

    print(f"Selected {len(cif_paths)} CIF(s):")
    for p in cif_paths:
        print(f"  {Path(p).name}")
    print(f"Output directory: {output_dir}\n")

    # ── Process each CIF ──────────────────────────────────────────────────
    dfs, failed = [], []

    for cif_path in cif_paths:
        name = Path(cif_path).name
        print(f"Processing: {name}")

        try:
            oxi = OXIDATION_STATES if OXIDATION_STATES else None
            s   = load_structure(cif_path, oxidation_states=oxi,
                                 data_dir=DATA_DIR)
            res = compute_maple(s, verbose=True)
            df  = results_to_dataframe(res)
            df["source_cif"] = name
            dfs.append(df)

        except Exception as e:
            failed.append((name, str(e)))
            print(f"  ✗ FAILED: {e}")
            print(f"    {traceback.format_exc().splitlines()[-1]}")

    # ── Save results ──────────────────────────────────────────────────────
    if dfs:
        master = pd.concat(dfs, ignore_index=True)

        # Site-level results
        results_csv = output_dir / 'maple_results.csv'
        master.to_csv(results_csv, index=False)

        # Summary table: one row per compound
        summary_cols = ["compound", "space_group", "Z", "d_ref_ang",
                        "madelung_factor", "total_maple", "disordered",
                        "source_cif"]
        summary = (master[summary_cols]
                   .drop_duplicates(subset="source_cif")
                   .reset_index(drop=True))
        summary_csv = output_dir / 'maple_summary.csv'
        summary.to_csv(summary_csv, index=False)

        print(f"\n{'='*60}")
        print(f"RESULTS: {len(dfs)} structure(s) processed")
        print(f"{'='*60}")
        print(f"Summary (per structure):")
        display(summary)
        print(f"\nSaved {len(master)} rows → {results_csv}")
        print(f"Saved {len(summary)} structures → {summary_csv}")
    else:
        master = None
        print("\nNo structures processed successfully.")

    # ── Save failures ─────────────────────────────────────────────────────
    if failed:
        failed_df = pd.DataFrame(failed, columns=['cif_file', 'error'])
        failed_csv = output_dir / 'maple_failed.csv'
        failed_df.to_csv(failed_csv, index=False)
        print(f"Saved {len(failed)} failures → {failed_csv}")

        print(f"\nFailed structures:")
        for name, err in failed:
            print(f"  {name}: {err}")

    # ── Summary statistics ────────────────────────────────────────────────
    n_total = len(cif_paths)
    n_ok    = len(dfs)
    n_fail  = len(failed)
    n_ordered    = sum(1 for d in dfs
                       if not d["disordered"].iloc[0]) if dfs else 0
    n_disordered = n_ok - n_ordered

    print(f"\n{'='*60}")
    print(f"  Total CIFs        : {n_total}")
    print(f"  Processed (OK)    : {n_ok}  "
          f"({n_ordered} ordered, {n_disordered} mean-field)")
    print(f"  Failed            : {n_fail}")
    print(f"{'='*60}")

    return master


# ── Launch ────────────────────────────────────────────────────────────────────
df_loaded = launch_maple_gui()

### Interpreting the GUI output

**Output files** (saved to `outputs/` subdirectory alongside the CIF files):

| File | Contents |
|------|----------|
| `maple_results.csv` | Full site-level data: one row per crystallographic site per structure. Use this for per-site analysis, BVS/MAPLE cross-comparison, and detailed diagnostics. |
| `maple_summary.csv` | One row per structure: compound, space group, Z, d_ref, MF, total MAPLE, disorder flag. Use this for database-scale statistics and the additivity analysis. |
| `maple_failed.csv` | One row per failed CIF: filename and error message. Use this for failure mode classification. |

**Column definitions (site-level, `maple_results.csv`):**

| Column | Meaning |
|--------|---------|
| `total_maple` | MAPLE per formula unit [kcal/mol]. For isostructural compounds, lower = longer bonds = larger cation. |
| `madelung_factor` | Dimensionless MF. Constant within a structure type (1.7476 for all rock salts). Deviations signal a topology change. |
| `d_ref_ang` | Shortest cation–anion distance [Å]. The reference distance on which PMC is normalised. |
| `pmc` | Partial Madelung constant. Geometrically invariant within a structure type; site-specific in lower-symmetry structures. |
| `reduced_maple` | MAPLE / z² / d_ref. Strips charge and distance dependence; enables cross-structure, cross-valence comparison. |
| `disordered` | `True` if any site in the structure has mixed occupancy. MAPLE calculated using mean-field charge approximation. |
| `mixed_site` | `True` for individual sites with mixed occupancy. The `element` column reports the majority species; `oxidation_state` reports the occupancy-weighted mean charge. |

**Diagnostic plots** (saved to `outputs/` as PNG):

| Plot | What it shows | What to look for |
|------|---------------|------------------|
| `plot_1_processing_overview.png` | Bar chart of ordered successes, mean-field successes, and failures by type. | Overall processing yield; ratio of ordered to disordered structures in the ICSD subset. |
| `plot_2_maple_vs_complexity.png` | Total MAPLE *vs.* atoms per formula unit, coloured by ordered/mean-field. | Strong linear correlation expected for well-behaved ionic compounds. Outliers flag unusual electrostatics or incorrect oxidation states. |
| `plot_3_madelung_factor_distribution.png` | Histogram of Madelung factors with NaCl reference line (1.7476). | Clustering by structure type; spread within a cluster indicates structural variation or errors. |

In [ ]:
# 8. Diagnostic Plots
# ─────────────────────────────────────────────────────────────────────────────
#
# Run this cell AFTER running the GUI (Section 7).
# It reads maple_summary.csv and maple_failed.csv from the outputs directory
# and generates three diagnostic plots, saved as PNG to the same directory.
#
# INSTRUCTIONS:
#   1. Set OUTPUT_DIR below to the 'outputs' folder created by the GUI.
#      (Copy the path printed by the GUI: "Output directory: ...")
#   2. Run this cell.

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
from pathlib import Path

# ── SET THIS PATH ─────────────────────────────────────────────────────────────
# Point to the PARENT folder containing your CIFs (the one with outputs_*
# subdirectories). The cell will automatically find the most recent run.
# Example: r"C:\Users\I6USER1\Downloads\MaplePy Test Batch 1"
CIF_DIR = r"C:\Users\I6USER1\Downloads\MaplePy Test Batch 1"
# ──────────────────────────────────────────────────────────────────────────────

cif_dir = Path(CIF_DIR)
output_dirs = sorted(cif_dir.glob('outputs_*'), reverse=True)
if output_dirs:
    output_dir = output_dirs[0]
    print(f"Using most recent output: {output_dir.name}")
else:
    # Fall back to plain 'outputs' from older runs
    output_dir = cif_dir / 'outputs'
    if not output_dir.exists():
        raise FileNotFoundError(
            f"No outputs_* folders found in {cif_dir}.\n"
            f"Run the GUI (Section 7) first."
        )
    print(f"Using: {output_dir}")

# ── Sheffield brand colours ───────────────────────────────────────────────────
SHEFFIELD = {
    'brand_violet' : '#440099',
    'dark_violet'  : '#24125E',
    'electric'     : '#7000FF',
    'powder_violet': '#A78CFA',
    'pale_violet'  : '#C8BBFF',
    'teal'         : '#005A8F',
    'aqua'         : '#00BBCC',
    'coral'        : '#E7004C',
    'flamingo'     : '#FF6371',
    'peach'        : '#FF9664',
    'peak_green'   : '#005750',
    'spearmint'    : '#3BD4AE',
    'midnight'     : '#131E29',
}

# ── Global plot styling ───────────────────────────────────────────────────────
plt.rcParams.update({
    'font.family'      : 'sans-serif',
    'font.size'        : 11,
    'axes.labelsize'   : 13,
    'axes.titlesize'   : 14,
    'figure.dpi'       : 150,
    'savefig.dpi'      : 300,
    'savefig.bbox'     : 'tight',
    'axes.linewidth'   : 1.0,
    'xtick.direction'  : 'in',
    'ytick.direction'  : 'in',
    'xtick.major.width': 1.0,
    'ytick.major.width': 1.0,
})


# ── Load data ─────────────────────────────────────────────────────────────────
summary_csv = output_dir / 'maple_summary.csv'
failed_csv  = output_dir / 'maple_failed.csv'
results_csv = output_dir / 'maple_results.csv'

if not summary_csv.exists():
    raise FileNotFoundError(
        f"No maple_summary.csv found in {output_dir}.\n"
        f"Run the GUI (Section 7) first, then set OUTPUT_DIR above."
    )

df_summary = pd.read_csv(summary_csv)
df_results = pd.read_csv(results_csv) if results_csv.exists() else None

if failed_csv.exists():
    df_failed = pd.read_csv(failed_csv)
else:
    df_failed = pd.DataFrame(columns=['cif_file', 'error'])

n_ordered    = (~df_summary['disordered']).sum()
n_disordered = df_summary['disordered'].sum()
n_failed     = len(df_failed)
n_total      = n_ordered + n_disordered + n_failed

print(f"Loaded: {len(df_summary)} successful structures, "
      f"{n_failed} failures, {n_total} total")


# ═════════════════════════════════════════════════════════════════════════════
# PLOT 1: Success / Failure Overview
# ═════════════════════════════════════════════════════════════════════════════

fig1, ax1 = plt.subplots(figsize=(8, 4))

# Classify failures
if len(df_failed) > 0:
    conditions = [
        df_failed['error'].str.contains('Mixed-occupancy', na=False),
        df_failed['error'].str.contains('not a valid Element', na=False),
        df_failed['error'].str.contains('Unrecognised element', na=False),
        df_failed['error'].str.contains('Invalid cif', na=False),
        df_failed['error'].str.contains('charge-neutral', na=False)
            | df_failed['error'].str.contains('NoneType', na=False),
    ]
    labels_fail = ['Mixed occupancy', 'Invalid element (D, M, ...)',
                   'Invalid CIF', 'Charge imbalance']

    # Count each failure type
    fail_counts = {}
    accounted = pd.Series(False, index=df_failed.index)
    for cond, label in zip(conditions, labels_fail):
        mask = cond & ~accounted
        fail_counts[label] = mask.sum()
        accounted = accounted | cond
    fail_counts['Other'] = (~accounted).sum()
    # Remove zero-count categories
    fail_counts = {k: v for k, v in fail_counts.items() if v > 0}
else:
    fail_counts = {}

# Build bar data
categories = []
counts     = []
colours    = []

categories.append('Ordered')
counts.append(int(n_ordered))
colours.append(SHEFFIELD['brand_violet'])

categories.append('Mean-field')
counts.append(int(n_disordered))
colours.append(SHEFFIELD['powder_violet'])

fail_colours = [SHEFFIELD['coral'], SHEFFIELD['peach'],
                SHEFFIELD['flamingo'], SHEFFIELD['aqua'],
                SHEFFIELD['spearmint']]
for i, (label, count) in enumerate(fail_counts.items()):
    categories.append(label)
    counts.append(count)
    colours.append(fail_colours[i % len(fail_colours)])

bars = ax1.barh(categories, counts, color=colours, edgecolor='white',
                linewidth=0.5)

# Add count labels on bars
for bar, count in zip(bars, counts):
    if count > 0:
        ax1.text(bar.get_width() + max(sum(counts)*0.01, 0.5),
                 bar.get_y() + bar.get_height()/2,
                 str(count), va='center', ha='left', fontsize=10,
                 color=SHEFFIELD['midnight'])

ax1.set_xlabel('Number of CIF files')
ax1.set_title('MaplePy Processing Results')
ax1.invert_yaxis()
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# Add percentage annotation
pct_ok = 100 * (n_ordered + n_disordered) / n_total if n_total > 0 else 0
ax1.text(0.98, 0.02, f'{pct_ok:.1f}% processed successfully',
         transform=ax1.transAxes, ha='right', va='bottom',
         fontsize=10, color=SHEFFIELD['dark_violet'], style='italic')

fig1.tight_layout()
fig1.savefig(output_dir / 'plot_1_processing_overview.png')
print(f"Saved: {output_dir / 'plot_1_processing_overview.png'}")
plt.show()


# ═════════════════════════════════════════════════════════════════════════════
# PLOT 2: Total MAPLE vs formula complexity
# ═════════════════════════════════════════════════════════════════════════════

if df_results is not None and len(df_summary) > 1:

    # Count atoms per formula unit from site-level data
    atoms_per_fu = (df_results.groupby('source_cif')
                    .apply(lambda g: len(g) / g['Z'].iloc[0],
                           include_groups=False)
                    .reset_index(name='atoms_per_fu'))

    df_plot = df_summary.merge(atoms_per_fu, on='source_cif', how='left')

    fig2, ax2 = plt.subplots(figsize=(7, 6))

    # Ordered vs mean-field
    mask_ord = ~df_plot['disordered']
    mask_dis = df_plot['disordered']

    ax2.scatter(df_plot.loc[mask_ord, 'atoms_per_fu'],
                df_plot.loc[mask_ord, 'total_maple'],
                c=SHEFFIELD['brand_violet'], s=25, alpha=0.7,
                edgecolors='white', linewidth=0.3,
                label=f'Ordered ({mask_ord.sum()})', zorder=3)

    if mask_dis.any():
        ax2.scatter(df_plot.loc[mask_dis, 'atoms_per_fu'],
                    df_plot.loc[mask_dis, 'total_maple'],
                    c=SHEFFIELD['powder_violet'], s=25, alpha=0.7,
                    edgecolors='white', linewidth=0.3, marker='D',
                    label=f'Mean-field ({mask_dis.sum()})', zorder=3)

    ax2.set_xlabel('Atoms per formula unit')
    ax2.set_ylabel('Total MAPLE / kcal mol$^{-1}$')
    ax2.set_title('MAPLE vs formula complexity')
    ax2.legend(frameon=False, fontsize=10)
    ax2.spines['top'].set_visible(False)
    ax2.spines['right'].set_visible(False)

    fig2.tight_layout()
    fig2.savefig(output_dir / 'plot_2_maple_vs_complexity.png')
    print(f"Saved: {output_dir / 'plot_2_maple_vs_complexity.png'}")
    plt.show()

else:
    print("Plot 2 skipped: need >1 structure in maple_summary.csv")


# ═════════════════════════════════════════════════════════════════════════════
# PLOT 3: Madelung factor distribution
# ═════════════════════════════════════════════════════════════════════════════

if len(df_summary) > 1:

    mf_vals = df_summary['madelung_factor'].dropna()

    fig3, ax3 = plt.subplots(figsize=(7, 5))

    ax3.hist(mf_vals, bins=min(50, max(10, len(mf_vals)//5)),
             color=SHEFFIELD['teal'], edgecolor='white', linewidth=0.5,
             alpha=0.85)

    ax3.set_xlabel('Madelung factor')
    ax3.set_ylabel('Count')
    ax3.set_title('Distribution of Madelung factors')
    ax3.spines['top'].set_visible(False)
    ax3.spines['right'].set_visible(False)

    # Annotate common structure types if recognisable peaks appear
    ax3.axvline(x=1.7476, color=SHEFFIELD['coral'], linestyle='--',
                linewidth=1, alpha=0.7)
    ax3.text(1.7476, ax3.get_ylim()[1]*0.95, ' NaCl\n (1.748)',
             fontsize=8, color=SHEFFIELD['coral'], va='top')

    fig3.tight_layout()
    fig3.savefig(output_dir / 'plot_3_madelung_factor_distribution.png')
    print(f"Saved: {output_dir / 'plot_3_madelung_factor_distribution.png'}")
    plt.show()

else:
    print("Plot 3 skipped: need >1 structure in maple_summary.csv")


print(f"\nAll plots saved to: {output_dir}")

---
## 9. Notes, Limitations & Future Work

### Limitations

**Disorder:** Mixed-occupancy sites are not handled. `pymatgen`'s `EwaldSummation`
requires integer oxidation states at each site. The correct extension is a mean-field
approximation: compute the weighted-average charge at each mixed site, analogous
to the WAGII extension of GII for BVS (Reeves-McLaren 2025, *Acta Cryst. B*).

**Molecular anions:** The additivity theorem breaks down for compounds containing
molecular-like anions (SO₄²⁻, PO₄³⁻, CO₃²⁻ etc.) where covalent bonding within
the anion is significant. Hoppe (1995) treats these using MAPLE "increments" derived
from well-characterised reference structures (e.g. MAPLE("SO₃") from sulphates).

**Reference distance:** `d_ref` is computed as the shortest cation–anion distance.
For direct quantitative comparison with Hoppe's tabulated values, the same CIF source
should be used. Differences in CIF geometry (different measurement conditions, year)
explain most discrepancies between calculated and Hoppe's MAPLE values.

### Integration with Eir
- Eir v1.4.2 uses gemmi for CIF parsing; MaplePy uses pymatgen.
- Integration path: add a `--maple` flag to Eir CLI that calls `compute_maple()`
  and appends MAPLE columns to the existing site-level CSV output.
- Column names follow Eir conventions: `lowercase_snake_case`, `_kcal_mol` suffix.

### Future extensions
- MAPLE-WAGII: occupancy-weighted partial MAPLE for disordered structures
- MEFIR (Mean Fictive Ionic Radii, Hoppe & Meyer 1976): back-calculated from PMC values
- Database-scale batch processing with ICSD export
- BVS vs MAPLE correlation analysis for systematic parameter inadequacy diagnosis

### Citation
When publishing results calculated with MaplePy:
> MAPLE calculations were performed using MaplePy (Reeves-McLaren, N., 2026),
> following the formalism of Hoppe (1966, *Angew. Chem. Int. Ed.* **5**, 95)
> and Hoppe (1995, *Z. Naturforsch. A* **50**, 555).